# Publish contend based on one model

This notebook contains the steps to generate the SSOT from an ODM model and render content.

This notebook will be executed by the run.bat

In [1]:
odm_source_folder = 'IM'
destination_folder = 'content'

# Skip generation of the SSOT from ODM? 
skip_ssot_generation = False

# Location of the python tools
tools_path = 'pythonWork/pythonSource'

In [2]:
VERSION_TAG = "2.7-development"

In [3]:
import sys
import os
from pathlib import Path
import glob
import json

In [4]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/generator-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

In [5]:
sys.argv[0]

'/opt/homebrew/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py'

In [6]:
import argparse

parser = argparse.ArgumentParser(description='Generate SSOT and diagrams from ODM model')
parser.add_argument('--model', '-m', dest='source_folder', default=odm_source_folder)
parser.add_argument('--destination', dest='destination_folder', default=destination_folder)
parser.add_argument('--skip-odm', '-n', action='store_true', dest='skip_odm',
                    help="Skip generation/update of single point of documentation (SPOD) from ODM")
parser.add_argument('--skip-web', '-w', action='store_true', dest='skip_web',
                    help="Skip generation of static web content")
parser.add_argument('--confluence', '-c', action='store_true', dest='confluence',
                    help="Skip generation of Confluence content")
parser.add_argument('--sharepoint', '-s', action='store_true', dest='sharepoint',
                    help="Enable generation of Sharepoint content")
parser.add_argument('--sparx-ea', '-e', action='store_true', dest='sparx_ea',
                    help="Enable generation Sparx Enterprise Architect compatible UML/XMI export")
parser.add_argument('--dont-merge', action="store_true", dest="clean_slate",
                    help="Do not merge! Existing database is renamed to .bkp")
parser.add_argument('--all', action="store_true", dest="all", help="Generate all possible outputs")
parser.add_argument('--tools-path', dest='tools_path', default='pythonWork/pythonSource', help="Tools path")
parser.add_argument('--profile', action='store_true', dest='profile', help="Enable profiling")
parser.add_argument('--languages', '-l', dest='languages', default='de', help="Comma separated list of languages (de,en,fr, ...)") 
parser.add_argument('--version', action='store_true')

arguments = argparse.Namespace()

if len(sys.argv) > 0 and '.py' in sys.argv[0] and not 'ipykernel' in sys.argv[0]:
    arguments = parser.parse_args()
    odm_source_folder = arguments.source_folder
    destination_folder = arguments.destination_folder
    skip_ssot_generation = arguments.skip_odm
    tools_path = arguments.tools_path
    if arguments.version:
        print(f"Version {VERSION_TAG}")
        exit(0)
else:
    # in jupyter environment
    assert 'ipykernel' in sys.argv[0], f"Not running in Jupyter environment 🙀"
    odm_source_folder = '../../pythonWork/pythonSource/testenvironment/testmodels/riddle/IM'
    destination_folder = 'content'
    tools_path = os.path.abspath('../../pythonWork/pythonSource')
    skip_ssot_generation = False
    arguments = argparse.Namespace(**{
        'clean_slate': False, 
        'sparx_ea': True, 
        'languages': 'de,en,fr', 
        'profile': True})


In [7]:
odm_source_folder = '/Users/bue/dev/fyyccim-tools/testdata/fyyccim-refmodels/CRM/IM'
odm_source_folder = '/Users/bue/projects/sika/Sika-IM/IM'

In [8]:
logger.debug(f"Configuration: {arguments}")

In [9]:
logger.info(f"Starting generator version {VERSION_TAG}")

INFO - Starting generator version 2.7-development


### Safeguards


In [10]:
models = glob.glob(odm_source_folder + '/*.[dD][mM][dD]')
if len(models) < 1:
    logger.fatal(f"No model file (*.dmd) found in source folder {os.path.abspath(odm_source_folder)}.")
    exit(2)

model = models[0]
if len(models) > 1:
    for name in models:
        # pick model with shortest name
        if len(name) < len(model):
            model = name
    logger.warning(f"Found {len(models)} models in {os.path.abspath(odm_source_folder)} using {model}")
    # disable other models?
    for name in models:
        if name != model:
            os.rename(name, os.path.splitext(name)[0] + '.hidden')

In [11]:
base_path = Path(odm_source_folder)
assert os.path.isdir(odm_source_folder), "Cannot find source folder {}".format(odm_source_folder)
project_files = list(base_path.glob('*.dmd'))
assert len(project_files) == 1, "Cannot find exactly 1 ODM .dmd file in source folder {}: {}".format(odm_source_folder,
                                                                                                     project_files)
logger.info(f"Processing the information model in {os.path.abspath(base_path)}")
from IPython.core.display import HTML

HTML(
    '<span style="font-family: Impact; font-size:48px">Processing the information model in<br/><span style="color: darkorange">{0}</span></span>'.format(
        os.path.abspath(odm_source_folder)))

INFO - Processing the information model in /Users/bue/projects/sika/Sika-IM/IM


In [12]:
if not (os.path.exists(tools_path)
        and os.path.isfile(os.path.join(tools_path, 'SSOT_db', 'createDB.py'))):
    logger.fatal(f"Tools not in expected path {tools_path}")
    exit(3)

logger.debug(f"Working with tools in {tools_path}")

# Add toolbox to python library path
sys.path.insert(0, os.path.abspath(tools_path))

from SSOT_infra import parameters
global parameters

from IM_WEB.IM_HTML import drawiodiagram
from SSOT_db.IM_JSON import JSModel

In [13]:
from SSOT_infra import logmessages

def tap_logmessages(message: str):
    logger.warning(message)


logmessages.logtrap = tap_logmessages

In [14]:
languages = [x.strip() for x in arguments.languages.split(',')]

# Process the datasource

In [24]:

model_path = Path(model).resolve()
model_name = model_path.stem
logger.info(f"Initializing parameters for model '{model_name}' in {model_path} for languages {languages}") 
parameters.initparam(pbasedirec=str(model_path.parent.resolve()), 
                     pdbfile=str(Path(model_path.parent.parent) / 'DB' / f"{model_name}.db"),
                     pmodelfilepath=str(model_path.parts[-2:]), 
                     pmodelname=model_name)

config_folder_before = None

odm_config_folder = Path(parameters.odmKonfDirec()).resolve()
if not os.path.isdir(odm_config_folder):
    logger.warning(f"Cannot find {odm_config_folder}")
    configuration_path = odm_config_folder.with_name('Configuration')
    parameters.odmKonfDirec(str(configuration_path))
    parameters.odmDefDomainsfilePath(str(Path(configuration_path, parameters.odmdefdomainsfile())))
    logger.warning(f"Patching config folder to {parameters.odmKonfDirec()}")
    if not configuration_path.is_dir():
        logger.warning("Missing configuration, patching ...")
        configuration_path.mkdir(exists_ok=True)

assert os.path.isdir(parameters.odmKonfDirec()), f"Configuration folder {parameters.odmKonfDirec()} not found"

INFO - Initializing parameters for model 'Sika-IM' in /Users/bue/projects/sika/Sika-IM/IM/Sika-IM.dmd for languages ['de', 'en', 'fr']
WARNING - Cannot find /Users/bue/projects/sika/Sika-IM/IM/Konfiguration
WARNING - Patching config folder to /Users/bue/projects/sika/Sika-IM/IM/Configuration


In [25]:
sqlfilepath = os.path.join(tools_path, 'IM_db/sqlfiles/')

In [26]:
# Ensure base folder exists
base = os.path.split(parameters.dbFilePath())[0]
os.makedirs(base, exist_ok=True)

In [27]:
if arguments.clean_slate and os.path.isfile(parameters.dbFilePath()):
    existing_database = Path(parameters.dbFilePath())
    logger.info(
        f"Don't merge existing SSOT {existing_database}. It is backed up as {existing_database.with_suffix('.bkp')}")
    existing_database.rename(existing_database.with_suffix('.bkp'))

In [28]:
from LOAD_MODELS.LOAD_ODM import fillDB
from LOAD_MODELS.LOAD_ODM import transferModel
from SSOT_db.createDB import upgradeDB
from SSOT_db.SQL_INFRA import dbConnect
from SSOT_db.IM_OBJECTS import Language
import atexit
import shutil

def merge():
    fillDB.fillmergedb(pdbfilepath=parameters.dbFilePath(),
                       createnewdb=not Path(parameters.dbFilePath()).is_file(),
                       transferfunction=transferModel.transferODMModel)

def rollback(backup: Path, destination: Path):
    crash = Path(parameters.dbFilePath())
    crash_report = Path(f'log/generator-{run_stamp}.db')
    logger.debug(f"Archiving crash db as {str(crash_report)}")
    crash.replace(crash_report)
    logger.info(f"Restoring {str(destination)} from backup {str(backup)}")    
    backup.replace(destination)
    
    
if not skip_ssot_generation:
    os.makedirs(parameters.dbDirect(), exist_ok=True)

    backup = None
    if not arguments.clean_slate and os.path.isfile(parameters.dbFilePath()):        
        backup = Path(parameters.dbFilePath()).parent / 'backup.db'
        logger.debug(f"Backing up database to {str(backup)}")
        shutil.copy(parameters.dbFilePath(), backup)
        atexit.register(rollback, backup=backup, destination=parameters.dbFilePath())
        logger.info('Updating database {db} from model {odm}'.format(db=parameters.dbFilePath(), odm=parameters.dbDirect()))
        try:
            applied = upgradeDB()
            if len(applied) > 0:
                logger.info(f"Updated {len(applied)} {applied}")
        except Exception as e:
            print("Cannot upgrade database {db}")
            raise e
    
    try:
        if os.path.isfile(parameters.dbFilePath()):
            # load languages from existing SSOT
            dbConnect.openDBbasic(parameters.dbFilePath(), '1')
            deflang = Language.getdefaultlang()
            if deflang.liesdeflangiso2() != arguments.languages.split(',')[0]:
                logger.warning(f"Default language in SSOT is '{deflang.liesdeflangiso2()}' but --language requests '{arguments.languages}'")
                logger.warning(f"Overriding default language from command line to '{deflang.liesdeflangiso2()}'")
            dbConnect.closeDB()
        
        parameters.dbLanguages(arguments.languages)
        parameters.dbDefaultLang(arguments.languages.split(',')[0])
        if arguments.profile:
            logger.info("Starting to profile fillDB.fillmergedb()")
            import cProfile
            profile_stats_file = Path(f'log/generator-{run_stamp}-fillmergedb.prof')
            cProfile.run('merge()', str(profile_stats_file))
            logger.info(f"Profiling complete. Result stored in {str(profile_stats_file)}")
        else:
            merge()
        logger.info(f"Sucessfully updated {parameters.dbFilePath()}")
    except:
        print('Consult logfile {} and {}'.format(logfile, parameters.logfilepath()))
        raise
    
    if backup is not None:
        logger.debug(f"Removing backup on success")
        atexit.unregister(rollback)
        backup.unlink()


INFO - Updating database /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.db from model /Users/bue/projects/sika/Sika-IM/DB
INFO - Starting to profile fillDB.fillmergedb()


DB /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.db is up to date: version 1.7


WARNING - dosSEGfiles: directory "/Users/bue/projects/sika/Sika-IM/IM/Sika-IM/rel/796FFEED-7AAD8B7A72CD//Users/bue/projects/sika/Sika-IM/IM/Sika-IM/table" not found.
WARNING - dosSEGfiles: directory "/Users/bue/projects/sika/Sika-IM/IM/Sika-IM/rel/20F6F28F-E7352E9A4E54//Users/bue/projects/sika/Sika-IM/IM/Sika-IM/table" not found.
WARNING - dosSEGfiles: directory "/Users/bue/projects/sika/Sika-IM/IM/Sika-IM/rel/E7A15F5E-3081C0B891C1//Users/bue/projects/sika/Sika-IM/IM/Sika-IM/table" not found.
WARNING - dosSEGfiles: directory "/Users/bue/projects/sika/Sika-IM/IM/Sika-IM/rel/C6E6FB8B-70285E00D02B//Users/bue/projects/sika/Sika-IM/IM/Sika-IM/table" not found.
WARNING - Mapping funktioniert nicht. (Table fehlt) :   Logic: type = 0   guid = 1B55D097-ADE8-574C-510A-1BFBBB688071    relational: type = 4   guid = EB8CCDF6-7A57-18FD-A468-E8928F7F64A3    file: /Users/bue/projects/sika/Sika-IM/IM/Sika-IM/mapping/ExtendedMap_RM20F6F28F-CB91-7AA5-0B97-E7352E9A4E54.xml
WARNING - Mapping funktioniert n

  0%|          | 0/78 [00:00<?, ?it/s]

INFO - Processing domains
INFO - Processing attributes
INFO - Processing diagrams


...

Errors 0,  Warnings 0
elements changed in database /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.db
          1321 inserted, 164 updated, 154 deleted, 0 references removed


  0%|          | 0/79 [00:00<?, ?it/s]

INFO - Processing domains
INFO - Processing attributes
INFO - Processing diagrams


0it [00:00, ?it/s]

INFO - Processing user defined properties
INFO - JSModel generated
INFO - Profiling complete. Result stored in log/generator-2022-02-07-11-57-58-fillmergedb.prof
INFO - Sucessfully updated /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.db


In [29]:
database_file = os.path.abspath(parameters.dbFilePath())
json_file = os.path.splitext(database_file)[0] + '.json'
assert os.path.isfile(json_file), f"SSOT file {json_file} not found"
logger.info(f"Loading SSOT from {json_file}")

INFO - Loading SSOT from /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json


In [30]:
with open(json_file) as f:
    data = json.load(f)
assert len(data) > 0, f'Config is empty :-('

In [31]:
jsmodel = JSModel.readfromfile(pfilename=json_file)

In [32]:
import gettext


class Translator:
    """Translate strings"""
    logger = logging.getLogger("Translator")

    def __init__(self, language: str):
        self.language = language
        self.translator = gettext.translation('confluence-publisher', './locale', fallback=True, languages=[language])
        self.title_format = '{title} - [{language}]'
        self.logger = logging.getLogger("Translator " + language)

    def tr(self, element) -> str:
        if isinstance(element, dict):
            """If the value provided is a field containing translations, use them"""
            text = element.get(self.language)
            if text is None:  #and len(element.values()) > 0:
                result = list(element.values())[0]
                if result is not None:
                    self.logger.warning(f"No translation for {text}. Falling back to {result} from {str(element)}")
                    text = result
            if not text:
                return ''
            # Strip leading translation marker
            #if '*de* ' in text:
            #    text = text.replace('*de* ', '', 1)
            return text

        # Fallback to gettext if not a dict
        if isinstance(element, str):
            translated = self.translator.gettext(element)
            return translated

        self.logger.warning("Cannot translate element '{0}' of type {1}".format(element, type(element)))
        return None

    def gettext(self, text: str):
        result = self.translator.gettext(text)
        if result == text:
            self.logger.warning("No translation for {0}".format(text))
        return result

    def translator(self):
        return self.translator

    def title_language(self, title: str) -> str:
        """Create unique confluence page title per translation"""
        return self.title_format.format(title=title, language=self.language)

    def key_lang(self, key: str) -> str:
        return key + '-' + self.language

    def lang(self) -> str:
        return self.language

In [33]:
translators = {language: Translator(language) for language in data['languages']}
translators

{'de': <__main__.Translator at 0x1661aa610>,
 'en': <__main__.Translator at 0x1661aa3d0>,
 'fr': <__main__.Translator at 0x165f1b520>}

In [34]:
def sanitize_filename(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ('.', '-', '_', ' ')).rstrip()

# Render draw.io diagrams

In [35]:
from lxml import etree

from tqdm.autonotebook import tqdm

content_root = '.'

diagram_count = len(data['diagrams'].keys()) * len(translators)

path = os.path.join(destination_folder, 'diagrams')
folder = os.path.join(content_root, path)
os.makedirs(folder, exist_ok=True)
logger.info(f"Rendering diagrams to {os.path.abspath(folder)}")

with tqdm(total=diagram_count, dynamic_ncols=True, unit='Diagram') as pbar:
    for lang, translator in translators.items():
        for key, diagram in data['diagrams'].items():
            filename = f"{key}-{sanitize_filename(diagram['name'])}-{lang}.drawio"
            file = os.path.join(folder, filename)

            pbar.set_description(f"Generating diagram {key} '{diagram['name']}' [{lang}] to {file}")
            draw_io_xml = drawiodiagram.create_diagram(key, jsmodel, translator)
            with open(file, 'wb') as out:
                out.write(etree.tostring(draw_io_xml))
                logger.debug(f"Diagram '{diagram['name']}' stored in draw.io format to {filename}")
            pbar.update(1)

INFO - Rendering diagrams to /Users/bue/dev/fyyccim-tools-integration/notebooks/mig/content/diagrams


  0%|                                                                                                         …

WARNING - Missing coordinates for label '' on start of relation RELA375
WARNING - Missing coordinates for label '' on end of relation RELA375
WARNING - Missing coordinates for label '' on start of relation RELA551
WARNING - Missing coordinates for label '' on end of relation RELA551
WARNING - Missing coordinates for label '' on start of relation RELA373
WARNING - Missing coordinates for label '' on end of relation RELA373
WARNING - Missing coordinates for label '' on start of relation RELA341
WARNING - Missing coordinates for label '' on end of relation RELA341
WARNING - Missing coordinates for label '' on start of relation RELA378
WARNING - Missing coordinates for label '' on end of relation RELA378
WARNING - Missing coordinates for label '' on start of relation RELA354
WARNING - Missing coordinates for label '' on end of relation RELA354
WARNING - Missing coordinates for label '' on start of relation RELA339
WARNING - Missing coordinates for label '' on end of relation RELA339
WARNIN

In [36]:
if config_folder_before is not None:
    logger.warning(f"Moving configuration folder back to {config_folder_before}")
    os.rename(odm_config_folder, config_folder_before)

In [37]:
logger.info(f"Successfully created {diagram_count} diagrams to {os.path.join(destination_folder, 'diagrams')}")

INFO - Successfully created 6 diagrams to content/diagrams


# Create web content

In [38]:
from IM_WEB import listWebdoku
from IM_WEB.IM_HTML.printHTML import HTMLExport
from SSOT_db.IM_OBJECTS import Languagetext

html_export = HTMLExport()
html_export.setmodel(jsmodel)

repo_root = os.getcwd()

if not os.path.isdir(os.path.join(repo_root, 'pythonWork')):
    repo_root = os.path.join(repo_root, '..', '..')

repo_root = os.path.abspath(os.path.join(repo_root, 'pythonWork', 'pythonSource', 'IM_WEB', 'html-lib'))
html_export.setWebDirec(destination_folder)
logger.info(f"Web ressources {html_export.libSourceDirec}")
listWebdoku.listwebmain(html_export, pfilter=None)

INFO - Web ressources /Users/bue/dev/fyyccim-tools-integration/pythonWork/pythonSource/IM_WEB/html-lib
INFO - Generating web content for language de in content/Sika-IM_de.html
INFO - Generating web content for language en in content/Sika-IM_en.html
INFO - Generating web content for language fr in content/Sika-IM_fr.html
INFO - Generating web content fo system CXM Access in content/CXM Access.html
INFO - Generating web content fo system CXM DPB Schnittstelle in content/CXM DPB Schnittstelle.html
INFO - Generating web content fo system CXM ProductUp FR in content/CXM ProductUp FR.html
INFO - Generating web content fo system SAP-ERP in content/SAP-ERP.html


# Sparx Enterprise Architect Export
Creates a XMI (XMI 2.1 / UML 2.1) export 
suitable to import the model into Sparx Enterprise Architect. 

In [39]:
from IM_EA.export.xmiexport import XMIBuilder
from lxml import etree

if arguments.sparx_ea or arguments.all:
    exporter = XMIBuilder(jsmodel, languages[0])
    tree = exporter.model_to_basic_xmi()
    exporter.model_to_ea_extension()

    destination = os.path.join(destination_folder, f"{data['model']['name']}.xmi")

    et = etree.ElementTree(tree)
    et.write(destination, pretty_print=True)
    logger.info(f"Exported for Sparx Enterprise Architect to {os.path.abspath(destination)}. Language '{languages[0]}'")

INFO - Exported for Sparx Enterprise Architect to /Users/bue/dev/fyyccim-tools-integration/notebooks/mig/content/Sika-IM.xmi. Language 'de'


# Report complete

In [40]:
print(f"\x1b[32mSucessfully\x1b[39m produced documentation in {os.path.abspath(destination_folder)}")

Sucessfully produced documentation in /Users/bue/dev/fyyccim-tools-integration/notebooks/mig/content
